# Eksperimen 6: Station-Wise Normalized Ensemble
**Strategi Utama: Normalisasi Target per Pos Pantau**

Berdasarkan diagnosa K-Fold CV Eksperimen 5 yang menghasilkan varians ekstrem (Fold 1: 5.36 vs Fold 5: 1.18), ditemukan akar masalah berupa heterogenitas baseline TMA antar pos pantau. Sungai di hulu pegunungan dan sungai di hilir pesisir memiliki level TMA yang berbeda hingga ratusan meter.

Solusi: Normalisasi target per pos pantau menggunakan statistik historis dari data *train*, melatih model pada ruang target yang seragam, lalu mengembalikan prediksi ke skala TMA absolut.

In [1]:
import pandas as pd
import numpy as np
import warnings
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error
from sklearn.cluster import KMeans
from sklearn.model_selection import TimeSeriesSplit
import optuna
import os

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 150)
optuna.logging.set_verbosity(optuna.logging.WARNING)

## 1. Pemuatan Data
Memuat seluruh data kompetisi: primer (Train/Test), lingkungan atmosfer-hidrologi, dan koordinat geografis pos pantau.

In [2]:
train = pd.read_csv('../data/raw/train.csv')
test = pd.read_csv('../data/raw/test.csv')
env_data = pd.read_csv('../data/raw/data_pendukung/data_lingkungan.csv')
coords = pd.read_csv('../data/raw/data_pendukung/koordinat_pos.csv')

train['datetime'] = pd.to_datetime(train['datetime'])
test['datetime'] = pd.to_datetime(test['id'].str[:19])
test['nama_pos'] = test['id'].str[22:]
env_data['datetime'] = pd.to_datetime(env_data['datetime'])

## 2. Exploratory Data Analysis (EDA)
Inspeksi dimensi, anomali data, dan batas kronologis matriks kompetisi.

In [3]:
print("=== Dimensi Matriks ===")
print("Train:", train.shape)
print("Test:", test.shape)
print("Lingkungan:", env_data.shape)

print("\n=== Cek Anomali Kosong (Missing Values) ===")
print(env_data.isnull().sum()[env_data.isnull().sum() > 0])

print("\n=== Batas Kronologis ===")
print("Akhir Train :", train['datetime'].max())
print("Awal Test   :", test['datetime'].min())
print("Selisih     :", test['datetime'].min() - train['datetime'].max())

=== Dimensi Matriks ===
Train: (84396, 3)
Test: (21780, 3)
Lingkungan: (888480, 27)

=== Cek Anomali Kosong (Missing Values) ===
soil_moisture_0_7cm          720
soil_moisture_7_28cm         720
soil_moisture_28_100cm       720
soil_moisture_100_255cm      720
surface_pressure_hpa         720
pressure_msl_hpa             720
rmm1                         720
rmm2                         720
mjo_phase                    720
mjo_amplitude                720
mjo_active                   720
nino_34                    12960
dtype: int64

=== Batas Kronologis ===
Akhir Train : 2025-09-18 18:00:00
Awal Test   : 2025-09-19 06:00:00
Selisih     : 0 days 12:00:00


### 2.1 Confirmatory EDA: Distribusi TMA per Pos Pantau
Memverifikasi bahwa setiap pos memiliki baseline TMA yang sangat berbeda. Inilah sumber masalah K-Fold varians tinggi.

In [4]:
station_stats = train.groupby('nama_pos')['tma_mdpl'].agg(['mean', 'std', 'min', 'max'])
station_stats = station_stats.sort_values('mean', ascending=False)
print("=== Distribusi TMA per Pos Pantau ===")
print(station_stats.round(2).to_string())

=== Distribusi TMA per Pos Pantau ===
                             mean   std     min     max
nama_pos                                               
Ngadipiro                  143.56  0.33  143.18  146.31
Ngrembang                  140.05  0.29  139.85  143.79
Wonogiri Dam               132.34  3.30  125.54  137.36
Badegan                    122.40  0.31  121.90  123.96
Colo Weir                  107.79  1.12  102.19  109.70
Kali Pepe - Tugu Boto       94.82  0.45   94.35  100.42
Peren                       91.31  2.02   90.11  170.10
Jarum                       90.72  3.92   89.29  250.14
Sekayu                      87.27  0.66   86.61   92.32
Kali Anyar - Kreteg Abang   86.46  4.46   84.44  323.21
Serenan                     86.32  0.70   85.47   90.98
Kali Pepe - PTPN            82.55  2.00    0.00  138.07
Jurug                       78.82  1.94    0.00   86.46
Kedungupit                  64.87  3.08   62.13  207.74
Kajangan                    51.04  2.61   49.18  172.77
Ketonggo  

### 2.2 CEDA: Korelasi Variabel Lingkungan terhadap TMA
Mengkonfirmasi variabel atmosfer mana yang berkorelasi paling kuat dengan pergerakan muka air.

In [5]:
train_temp = pd.merge(train, env_data, on=['datetime', 'nama_pos'], how='left')
num_cols = train_temp.select_dtypes(include=[np.number]).columns
corr = train_temp[num_cols].corr()['tma_mdpl'].sort_values(ascending=False)
print("=== Top Korelasi terhadap TMA ===")
print(corr.head(6).round(4))
print("...")
print(corr.tail(5).round(4))

=== Top Korelasi terhadap TMA ===
tma_mdpl                   1.0000
soil_moisture_100_255cm    0.1894
built_surface_m2           0.1793
soil_moisture_28_100cm     0.1270
soil_moisture_7_28cm       0.1238
soil_moisture_0_7cm        0.1058
Name: tma_mdpl, dtype: float64
...
rainfall_max_24h_mm    -0.0251
temperature_c          -0.0770
dew_point_c            -0.1217
landcover_class        -0.1573
surface_pressure_hpa   -0.9474
Name: tma_mdpl, dtype: float64


## 3. Station-Wise Statistics (Anti-Leakage)
Menghitung statistik deskriptif TMA **murni dari data Train** untuk digunakan sebagai fitur dan sebagai denormalizer prediksi. Statistik dari data Test tidak boleh digunakan di tahap ini.

In [6]:
station_profile = train.groupby('nama_pos')['tma_mdpl'].agg(
    tma_mean='mean',
    tma_std='std',
    tma_p25=lambda x: x.quantile(0.25),
    tma_p75=lambda x: x.quantile(0.75)
).reset_index()

station_profile['tma_std'] = station_profile['tma_std'].fillna(1.0)

global_mean = train['tma_mdpl'].mean()
global_std = train['tma_mdpl'].std()

print("Profil Statistik per Pos (Sampel 5 Teratas):")
print(station_profile.sort_values('tma_mean', ascending=False).head(5).to_string(index=False))

Profil Statistik per Pos (Sampel 5 Teratas):
    nama_pos   tma_mean  tma_std    tma_p25    tma_p75
   Ngadipiro 143.562796 0.334947 143.330000 143.710000
   Ngrembang 140.050592 0.286869 139.928190 140.051060
Wonogiri Dam 132.340343 3.299649 129.884407 135.097395
     Badegan 122.402005 0.309940 122.180000 122.560000
   Colo Weir 107.789569 1.119431 107.800000 108.260000


## 4. Preprocessing Adaptif
Imputasi berbeda untuk dua kategori data: variabel sensor atmosfer kontinu diinterpolasi secara linear, sedangkan indeks iklim makro global yang jarang berubah diisi secara *forward fill*.

In [7]:
env_data = env_data.sort_values(['nama_pos', 'datetime'])

macro_cols = ['nino_34', 'mjo_phase', 'mjo_amplitude', 'mjo_active', 'rmm1', 'rmm2']
dynamic_cols = ['surface_pressure_hpa', 'pressure_msl_hpa', 'soil_moisture_0_7cm',
                'soil_moisture_7_28cm', 'soil_moisture_28_100cm', 'soil_moisture_100_255cm']

for c in macro_cols:
    env_data[c] = env_data.groupby('nama_pos')[c].ffill().bfill()

for c in dynamic_cols:
    env_data[c] = env_data.groupby('nama_pos')[c].apply(
        lambda x: x.interpolate(method='linear').bfill().ffill()
    ).reset_index(level=0, drop=True)

### 4.1 Agregasi Frekuensi & Penggabungan Geospasial
Sensor atmosfer 1-jam diselaraskan ke resolusi 3-jam. Koordinat geografis dan klaster spasial digabungkan ke matriks utama.

In [8]:
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
coords['spatial_cluster'] = kmeans.fit_predict(coords[['latitude', 'longitude']])

def aggregate_env_data(df):
    agg_funcs = {col: 'mean' for col in df.columns if col not in ['nama_pos', 'landcover_name', 'datetime']}
    agg_funcs['rainfall_mm'] = 'sum'
    agg_funcs['rainfall_openmeteo_mm'] = 'sum'
    agg_funcs['rainfall_max_24h_mm'] = 'max'
    df_indexed = df.set_index('datetime')
    return df_indexed.groupby(['nama_pos', pd.Grouper(freq='3h', label='right', closed='right')]).agg(agg_funcs).reset_index()

env_agg = aggregate_env_data(env_data)

test['tma_mdpl'] = np.nan
all_data = pd.concat([train, test], ignore_index=True)
all_data = all_data.sort_values(['nama_pos', 'datetime']).reset_index(drop=True)

all_data = pd.merge(all_data, env_agg, on=['datetime', 'nama_pos'], how='left')
all_data = pd.merge(all_data, coords, on='nama_pos', how='left')
all_data = pd.merge(all_data, station_profile, on='nama_pos', how='left')

all_data['tma_mean'] = all_data['tma_mean'].fillna(global_mean)
all_data['tma_std'] = all_data['tma_std'].fillna(global_std)

## 5. Deep Feature Engineering
Rekayasa fitur fisika hidrologi secara masif. Rolling windows yang panjang mensimulasikan perambatan massa air dari hulu ke hilir Bengawan Solo.

In [9]:
le = LabelEncoder()
all_data['nama_pos_encoded'] = le.fit_transform(all_data['nama_pos'])

all_data['month'] = all_data['datetime'].dt.month
all_data['hour'] = all_data['datetime'].dt.hour
all_data['day_of_year'] = all_data['datetime'].dt.dayofyear
all_data['sin_hour'] = np.sin(2 * np.pi * all_data['hour'] / 24)
all_data['cos_hour'] = np.cos(2 * np.pi * all_data['hour'] / 24)
all_data['sin_month'] = np.sin(2 * np.pi * all_data['month'] / 12)
all_data['cos_month'] = np.cos(2 * np.pi * all_data['month'] / 12)

all_data['runoff_factor'] = all_data['rainfall_mm'] * all_data['soil_moisture_0_7cm']
all_data['pressure_drop'] = all_data.groupby('nama_pos')['surface_pressure_hpa'].diff(1).fillna(0)
all_data['rainfall_intensity'] = all_data['rainfall_mm'] / (all_data['soil_moisture_0_7cm'] + 1e-6)

windows = [4, 8, 24, 56]
for w in windows:
    all_data[f'rainfall_roll_{w}'] = all_data.groupby('nama_pos')['rainfall_mm'].transform(
        lambda x: x.rolling(window=w, min_periods=1).sum()
    )
    all_data[f'soil_roll_{w}'] = all_data.groupby('nama_pos')['soil_moisture_0_7cm'].transform(
        lambda x: x.rolling(window=w, min_periods=1).mean()
    )
    all_data[f'pressure_roll_{w}'] = all_data.groupby('nama_pos')['surface_pressure_hpa'].transform(
        lambda x: x.rolling(window=w, min_periods=1).mean()
    )

## 6. Target Normalization
Mentransformasikan target TMA dari ruang absolut ke ruang ternormalisasi per pos. Model akan belajar memprediksi deviasi dari baseline pos, bukan nilai absolut yang heterogen.

In [10]:
train_mask = all_data['tma_mdpl'].notnull()

all_data['tma_normalized'] = np.nan
all_data.loc[train_mask, 'tma_normalized'] = (
    (all_data.loc[train_mask, 'tma_mdpl'] - all_data.loc[train_mask, 'tma_mean']) /
    all_data.loc[train_mask, 'tma_std']
)

print("Distribusi target setelah normalisasi:")
print(all_data.loc[train_mask, 'tma_normalized'].describe().round(4))

Distribusi target setelah normalisasi:
count    84396.0000
mean         0.0000
std          0.9998
min        -41.2179
25%         -0.4486
50%         -0.1218
75%          0.2773
max         53.0309
Name: tma_normalized, dtype: float64


## 7. Training Setup: K-Fold Time-Series Validation
Membangun matriks pelatihan dan perlindungan 5-lipatan waktu (Walk-Forward). Target yang dipakai adalah `tma_normalized`.

In [11]:
train_data = all_data[train_mask].sort_values('datetime').reset_index(drop=True)
test_data = all_data[~train_mask].sort_values('datetime').reset_index(drop=True)

drop_cols = ['datetime', 'nama_pos', 'tma_mdpl', 'tma_normalized', 'id', 'landcover_name']
features = [c for c in train_data.columns if c not in drop_cols]
target = 'tma_normalized'

X_full = train_data[features]
y_full = train_data[target]
X_test = test_data[features]

tscv = TimeSeriesSplit(n_splits=5)
print(f"Total fitur yang digunakan: {len(features)}")
print("Daftar fitur:", features)

Total fitur yang digunakan: 54
Daftar fitur: ['rainfall_mm', 'humidity_pct', 'wind_direction_deg', 'dew_point_c', 'cloud_cover_pct', 'temperature_c', 'wind_speed_kmh', 'rainfall_openmeteo_mm', 'rainfall_max_24h_mm', 'solar_radiation_mj_m2', 'soil_moisture_0_7cm', 'soil_moisture_7_28cm', 'soil_moisture_28_100cm', 'soil_moisture_100_255cm', 'surface_pressure_hpa', 'pressure_msl_hpa', 'built_surface_m2', 'landcover_class', 'rmm1', 'rmm2', 'mjo_phase', 'mjo_amplitude', 'mjo_active', 'nino_34', 'latitude', 'longitude', 'spatial_cluster', 'tma_mean', 'tma_std', 'tma_p25', 'tma_p75', 'nama_pos_encoded', 'month', 'hour', 'day_of_year', 'sin_hour', 'cos_hour', 'sin_month', 'cos_month', 'runoff_factor', 'pressure_drop', 'rainfall_intensity', 'rainfall_roll_4', 'soil_roll_4', 'pressure_roll_4', 'rainfall_roll_8', 'soil_roll_8', 'pressure_roll_8', 'rainfall_roll_24', 'soil_roll_24', 'pressure_roll_24', 'rainfall_roll_56', 'soil_roll_56', 'pressure_roll_56']


### 7.1 Optuna Tuning: LightGBM
Mencari konfigurasi hyperparameter LightGBM terbaik berdasarkan rata-rata K-Fold RMSE pada target ternormalisasi.

In [12]:
def objective_lgb(trial):
    params = {
        'n_estimators': 500,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 31, 255),
        'max_depth': trial.suggest_int('max_depth', 5, 12),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 100),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'random_state': 42,
        'verbose': -1
    }
    scores = []
    for train_idx, val_idx in tscv.split(X_full):
        X_tr, X_va = X_full.iloc[train_idx], X_full.iloc[val_idx]
        y_tr, y_va = y_full.iloc[train_idx], y_full.iloc[val_idx]
        m = lgb.LGBMRegressor(**params)
        m.fit(X_tr, y_tr)
        scores.append(mean_squared_error(y_va, m.predict(X_va)))
    return np.mean(scores)

print("Memulai Optuna Tuning LightGBM (30 trials)...")
study_lgb = optuna.create_study(direction='minimize')
study_lgb.optimize(objective_lgb, n_trials=30)
best_lgb = study_lgb.best_params
best_lgb.update({'n_estimators': 1000, 'random_state': 42, 'verbose': -1})
print("Best LightGBM Params:", best_lgb)

Memulai Optuna Tuning LightGBM (30 trials)...
Best LightGBM Params: {'learning_rate': 0.02545589584033191, 'num_leaves': 193, 'max_depth': 5, 'subsample': 0.8878385718408531, 'colsample_bytree': 0.6498010154783648, 'min_child_samples': 77, 'reg_alpha': 0.0017684457110113906, 'reg_lambda': 0.0001420128925571292, 'n_estimators': 1000, 'random_state': 42, 'verbose': -1}


### 7.2 Optuna Tuning: XGBoost

In [13]:
def objective_xgb(trial):
    params = {
        'n_estimators': 500,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 4, 10),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'random_state': 42
    }
    scores = []
    for train_idx, val_idx in tscv.split(X_full):
        X_tr, X_va = X_full.iloc[train_idx], X_full.iloc[val_idx]
        y_tr, y_va = y_full.iloc[train_idx], y_full.iloc[val_idx]
        m = xgb.XGBRegressor(**params, verbosity=0)
        m.fit(X_tr, y_tr)
        scores.append(mean_squared_error(y_va, m.predict(X_va)))
    return np.mean(scores)

print("Memulai Optuna Tuning XGBoost (30 trials)...")
study_xgb = optuna.create_study(direction='minimize')
study_xgb.optimize(objective_xgb, n_trials=30)
best_xgb = study_xgb.best_params
best_xgb.update({'n_estimators': 1000, 'random_state': 42, 'verbosity': 0})
print("Best XGBoost Params:", best_xgb)

Memulai Optuna Tuning XGBoost (30 trials)...
Best XGBoost Params: {'learning_rate': 0.016595516390060183, 'max_depth': 4, 'subsample': 0.7934870031032826, 'colsample_bytree': 0.7306715692037572, 'min_child_weight': 2, 'reg_alpha': 0.15738032464948662, 'reg_lambda': 9.355722284328161, 'n_estimators': 1000, 'random_state': 42, 'verbosity': 0}


## 8. K-Fold Triple Ensemble Training
Melatih Triple Ensemble (LightGBM, XGBoost, CatBoost) melintasi 5 lipatan waktu. Setiap fold melaporkan RMSE dalam satuan TMA absolut (setelah denormalisasi) agar sebanding dengan skor Kaggle.

In [14]:
cat_params = {
    'iterations': 1000,
    'learning_rate': 0.03,
    'depth': 8,
    'random_seed': 42,
    'verbose': False
}

test_preds_lgb = np.zeros(len(X_test))
test_preds_xgb = np.zeros(len(X_test))
test_preds_cat = np.zeros(len(X_test))
cv_rmse_scores = []

print("Memulai K-Fold Triple Ensemble Training...")
for fold, (train_idx, val_idx) in enumerate(tscv.split(X_full)):
    X_tr, X_va = X_full.iloc[train_idx], X_full.iloc[val_idx]
    y_tr, y_va = y_full.iloc[train_idx], y_full.iloc[val_idx]

    val_mean = train_data.iloc[val_idx]['tma_mean'].values
    val_std = train_data.iloc[val_idx]['tma_std'].values

    m_lgb = lgb.LGBMRegressor(**best_lgb)
    m_xgb = xgb.XGBRegressor(**best_xgb)
    m_cat = CatBoostRegressor(**cat_params)

    m_lgb.fit(X_tr, y_tr)
    m_xgb.fit(X_tr, y_tr)
    m_cat.fit(X_tr, y_tr)

    p_lgb_norm = m_lgb.predict(X_va)
    p_xgb_norm = m_xgb.predict(X_va)
    p_cat_norm = m_cat.predict(X_va)

    blend_norm = (p_lgb_norm * 0.35) + (p_xgb_norm * 0.35) + (p_cat_norm * 0.30)
    blend_abs = (blend_norm * val_std) + val_mean

    y_va_abs = (y_va.values * val_std) + val_mean
    fold_rmse = np.sqrt(mean_squared_error(y_va_abs, blend_abs))
    cv_rmse_scores.append(fold_rmse)
    print(f"Fold {fold+1} RMSE (TMA absolut): {fold_rmse:.4f}")

    test_preds_lgb += m_lgb.predict(X_test) / tscv.n_splits
    test_preds_xgb += m_xgb.predict(X_test) / tscv.n_splits
    test_preds_cat += m_cat.predict(X_test) / tscv.n_splits

print(f"\nRata-rata K-Fold CV RMSE (Denormalized): {np.mean(cv_rmse_scores):.4f}")

Memulai K-Fold Triple Ensemble Training...
Fold 1 RMSE (TMA absolut): 3.9578
Fold 2 RMSE (TMA absolut): 1.9646
Fold 3 RMSE (TMA absolut): 0.9241
Fold 4 RMSE (TMA absolut): 1.3364
Fold 5 RMSE (TMA absolut): 1.0739

Rata-rata K-Fold CV RMSE (Denormalized): 1.8514


## 9. Denormalisasi & Output Prediksi
Prediksi ternormalisasi dikembalikan ke skala TMA absolut menggunakan statistik per pos yang telah dihitung dari data *train*.

In [15]:
blend_norm_final = (test_preds_lgb * 0.35) + (test_preds_xgb * 0.35) + (test_preds_cat * 0.30)

test_mean = test_data['tma_mean'].values
test_std = test_data['tma_std'].values

final_tma = (blend_norm_final * test_std) + test_mean

test_data['tma_mdpl'] = final_tma
submission = test_data[['id', 'tma_mdpl']]

if not os.path.exists('../submissions'):
    os.makedirs('../submissions')

submission.to_csv('../submissions/submission.csv', index=False)
print("Eksperimen 6 selesai. File submission.csv telah tersimpan di submissions/.")
print(f"Rentang prediksi: {final_tma.min():.2f} - {final_tma.max():.2f}")

Eksperimen 6 selesai. File submission.csv telah tersimpan di submissions/.
Rentang prediksi: 0.67 - 144.67
